# Experiment 2: Maximum Likelihood Estimation (MLE) for Probability Distribution Parameter Estimation

## Objective
Implement **Maximum Likelihood Estimation (MLE)** on a synthetic dataset and estimate the parameters of a Normal distribution — then extend the concept to its role in **Generative AI**.

## What is MLE?

MLE answers the question:
> *"Given this data, what distribution parameters make this data most probable?"*

Given observed data $\mathbf{x} = \{x_1, x_2, \ldots, x_n\}$, MLE finds parameters $\theta$ that maximise the likelihood:

$$\hat{\theta}_{MLE} = \arg\max_{\theta} \ L(\theta \mid \mathbf{x}) = \arg\max_{\theta} \prod_{i=1}^{n} p(x_i \mid \theta)$$

Since products are numerically unstable, we maximise the **log-likelihood** instead:

$$\hat{\theta}_{MLE} = \arg\max_{\theta} \sum_{i=1}^{n} \log p(x_i \mid \theta)$$

## Why MLE Matters in Generative AI

| GenAI Model | How MLE is Used |
|---|---|
| **Large Language Models (GPT, Claude)** | Training = maximising log-likelihood of next token given context |
| **Variational Autoencoders (VAE)** | ELBO objective contains a reconstruction log-likelihood term |
| **Gaussian Mixture Models (GMM)** | EM algorithm alternates between E-step and MLE M-step |
| **Diffusion Models** | Score matching is equivalent to a weighted MLE over noise levels |
| **GANs** | Discriminator minimises cross-entropy = a form of MLE |

> **Key Insight:** Every time a GenAI model is trained on data, it is performing MLE (or a variation like MAP) — learning the parameters that make the training data most probable under the model.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

np.random.seed(42)
print('Libraries loaded. Ready to perform MLE.')

---
## Step 1: Generate Synthetic Data

We simulate a dataset from a **known** Normal distribution. In practice (and in GenAI), the true parameters are unknown — MLE's job is to recover them from raw data.

Think of this as: a GenAI model observing thousands of training sentences and trying to infer the underlying language distribution.

In [ ]:
# True parameters (hidden from the model — only the data is observed)
true_mu    = 50
true_sigma = 10
N = 1000

# Generate synthetic dataset
data = np.random.normal(true_mu, true_sigma, N)

print(f'Dataset: {N} samples from N(mu={true_mu}, sigma={true_sigma})')
print(f'Sample Mean : {np.mean(data):.4f}')
print(f'Sample Std  : {np.std(data):.4f}')
print(f'Min / Max   : {data.min():.2f} / {data.max():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(data, bins=35, density=True, alpha=0.65, color='steelblue', label='Synthetic Data')
x_range = np.linspace(data.min(), data.max(), 300)
axes[0].plot(x_range, norm.pdf(x_range, true_mu, true_sigma), 'r--',
             linewidth=2, label=f'True PDF N({true_mu}, {true_sigma})')
axes[0].set_title('Synthetic Dataset — Histogram & True PDF\n(In GenAI: this is your training data)', fontsize=12)
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Density')
axes[0].legend()

# Box plot to show spread
axes[1].boxplot(data, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Box Plot — Data Spread\nOutliers shown as individual dots', fontsize=12)
axes[1].set_ylabel('Value')
axes[1].set_xticks([])

plt.suptitle('Step 1: Observe the Data (True Parameters are Unknown to the Model)',
             fontsize=11, style='italic', color='dimgray', y=1.02)
plt.tight_layout()
plt.show()

---
## Step 2: Define the Log-Likelihood Function

### The Normal Distribution PDF

$$f(x \mid \mu, \sigma) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

### Likelihood for n independent observations

$$L(\mu, \sigma \mid \mathbf{x}) = \prod_{i=1}^{n} f(x_i \mid \mu, \sigma)$$

### Log-Likelihood (numerically stable form)

$$\ln L(\mu, \sigma \mid \mathbf{x}) = -\frac{n}{2}\ln(2\pi) - n\ln(\sigma) - \frac{1}{2\sigma^2}\sum_{i=1}^{n}(x_i - \mu)^2$$

We **minimise the negative log-likelihood** because optimisers (like scipy's `minimize`) are designed for minimisation.

> **GenAI Parallel:** Training a neural language model means minimising the negative log-likelihood (cross-entropy loss) of the correct next token — the exact same objective, just over a categorical distribution.

In [ ]:
def negative_log_likelihood(params, data):
    """
    Negative Log-Likelihood for a Normal distribution.
    params = [mu, sigma]
    Equivalent to cross-entropy loss in neural language model training.
    """
    mu, sigma = params
    if sigma <= 0:
        return np.inf  # sigma must be positive
    log_likelihood = np.sum(norm.logpdf(data, loc=mu, scale=sigma))
    return -log_likelihood

# Visualise the log-likelihood surface over a grid of (mu, sigma) values
mu_range    = np.linspace(45, 55, 100)
sigma_range = np.linspace(7, 13, 100)
MU, SIGMA   = np.meshgrid(mu_range, sigma_range)

NLL = np.array([
    [negative_log_likelihood([m, s], data) for m in mu_range]
    for s in sigma_range
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contour plot of NLL surface
cp = axes[0].contourf(MU, SIGMA, NLL, levels=40, cmap='viridis')
plt.colorbar(cp, ax=axes[0], label='Negative Log-Likelihood')
axes[0].scatter([true_mu], [true_sigma], color='red', marker='*', s=200, label='True params', zorder=5)
axes[0].set_title('NLL Surface — Lower = Better Fit\nMinimum should be near true parameters', fontsize=12)
axes[0].set_xlabel('mu')
axes[0].set_ylabel('sigma')
axes[0].legend()

# Log-likelihood vs mu (with sigma fixed at true value)
ll_vs_mu = [-negative_log_likelihood([m, true_sigma], data) for m in mu_range]
axes[1].plot(mu_range, ll_vs_mu, color='steelblue', linewidth=2)
axes[1].axvline(true_mu, color='red', linestyle='--', linewidth=2, label=f'True mu={true_mu}')
axes[1].axvline(np.mean(data), color='green', linestyle=':', linewidth=2,
                label=f'Sample mean={np.mean(data):.2f}')
axes[1].set_title('Log-Likelihood vs mu (sigma fixed)\nPeak = MLE estimate for mu', fontsize=12)
axes[1].set_xlabel('mu')
axes[1].set_ylabel('Log-Likelihood')
axes[1].legend()

plt.suptitle('Step 2: The Log-Likelihood Landscape — MLE finds the peak',
             fontsize=11, style='italic', color='dimgray', y=1.02)
plt.tight_layout()
plt.show()

print('Negative Log-Likelihood function defined.')

---
## Step 3: Optimise — Find the MLE Parameters

We use **L-BFGS-B** (Limited-memory Broyden–Fletcher–Goldfarb–Shanno with Bounds), a gradient-based optimiser well suited for continuous parameter estimation.

### Analytical MLE solution for Normal distribution:
$$\hat{\mu}_{MLE} = \frac{1}{n}\sum_{i=1}^{n} x_i = \bar{x}$$
$$\hat{\sigma}_{MLE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

The numerical optimiser should converge to exactly these values — a great sanity check!

In [ ]:
# Initial guess: use sample statistics (warm start — same strategy as pre-training in GenAI)
initial_mu    = np.mean(data)
initial_sigma = np.std(data)
initial_params = [initial_mu, initial_sigma]

print('=== MLE Optimisation ===')
print(f'Initial guess  ->  mu: {initial_mu:.4f}  |  sigma: {initial_sigma:.4f}')

# Bounds: mu unrestricted, sigma > 0
bounds = [(None, None), (1e-6, None)]

# Run numerical optimisation
result = minimize(
    negative_log_likelihood,
    initial_params,
    args=(data,),
    method='L-BFGS-B',
    bounds=bounds
)

estimated_mu, estimated_sigma = result.x

# Analytical MLE (closed-form)
analytical_mu    = np.mean(data)
analytical_sigma = np.std(data)  # MLE sigma (divides by N, not N-1)

print(f'\nOptimisation Status : {result.message}')
print(f'Converged           : {result.success}')
print(f'Iterations          : {result.nit}')
print()
print(f'{"Parameter":<12} {"True":>10} {"MLE (Numerical)":>18} {"MLE (Analytical)":>18} {"Error %":>10}')
print('-' * 72)
print(f'{"mu":<12} {true_mu:>10.4f} {estimated_mu:>18.4f} {analytical_mu:>18.4f} {abs(estimated_mu - true_mu)/true_mu*100:>9.4f}%')
print(f'{"sigma":<12} {true_sigma:>10.4f} {estimated_sigma:>18.4f} {analytical_sigma:>18.4f} {abs(estimated_sigma - true_sigma)/true_sigma*100:>9.4f}%')

---
## Step 4: Visualise — MLE Fit vs True Distribution

In [ ]:
x = np.linspace(data.min() - 5, data.max() + 5, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Main fit comparison
axes[0].hist(data, bins=35, density=True, alpha=0.5, color='steelblue', label='Observed Data')
axes[0].plot(x, norm.pdf(x, estimated_mu, estimated_sigma), 'r-', linewidth=2.5,
             label=f'MLE Fit: N({estimated_mu:.2f}, {estimated_sigma:.2f})')
axes[0].plot(x, norm.pdf(x, true_mu, true_sigma), 'g--', linewidth=2.5,
             label=f'True:    N({true_mu}, {true_sigma})')
axes[0].set_title('MLE Estimated vs True Distribution\nCloser the curves, better the estimation', fontsize=12)
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Density')
axes[0].legend()

# Residuals: difference between MLE PDF and True PDF
residuals = norm.pdf(x, estimated_mu, estimated_sigma) - norm.pdf(x, true_mu, true_sigma)
axes[1].plot(x, residuals, color='purple', linewidth=2)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].fill_between(x, residuals, 0, alpha=0.2, color='purple')
axes[1].set_title('Residuals: MLE PDF - True PDF\nSmall residuals = accurate estimation', fontsize=12)
axes[1].set_xlabel('Value')
axes[1].set_ylabel('PDF Difference')

plt.suptitle('Step 4: MLE Fit Quality — How well did MLE recover the true parameters?',
             fontsize=11, style='italic', color='dimgray', y=1.02)
plt.tight_layout()
plt.show()

---
## Step 5: Effect of Sample Size on MLE Accuracy

A key property of MLE is **consistency** — as sample size $n \to \infty$, the MLE estimates converge to the true parameters. This mirrors how larger training datasets improve GenAI model quality.

In [ ]:
sample_sizes = [10, 50, 100, 500, 1000, 5000, 10000]
mle_mus, mle_sigmas = [], []

for n in sample_sizes:
    d = np.random.normal(true_mu, true_sigma, n)
    res = minimize(negative_log_likelihood, [np.mean(d), np.std(d)],
                   args=(d,), method='L-BFGS-B', bounds=[(None, None), (1e-6, None)])
    mle_mus.append(res.x[0])
    mle_sigmas.append(res.x[1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogx(sample_sizes, mle_mus, 'o-', color='steelblue', linewidth=2, markersize=7)
axes[0].axhline(true_mu, color='red', linestyle='--', linewidth=2, label=f'True mu = {true_mu}')
axes[0].set_title('MLE Estimate of mu vs Sample Size\nConverges to true value as n increases', fontsize=12)
axes[0].set_xlabel('Sample Size (log scale)')
axes[0].set_ylabel('Estimated mu')
axes[0].legend()
axes[0].set_xticks(sample_sizes)
axes[0].set_xticklabels(sample_sizes, rotation=30)

axes[1].semilogx(sample_sizes, mle_sigmas, 'o-', color='coral', linewidth=2, markersize=7)
axes[1].axhline(true_sigma, color='red', linestyle='--', linewidth=2, label=f'True sigma = {true_sigma}')
axes[1].set_title('MLE Estimate of sigma vs Sample Size\nMore data = more accurate estimation', fontsize=12)
axes[1].set_xlabel('Sample Size (log scale)')
axes[1].set_ylabel('Estimated sigma')
axes[1].legend()
axes[1].set_xticks(sample_sizes)
axes[1].set_xticklabels(sample_sizes, rotation=30)

plt.suptitle('Step 5: MLE Consistency — More Data = Better Parameter Recovery\n'
             '(GenAI: why larger training datasets improve model quality)',
             fontsize=11, style='italic', color='dimgray', y=1.02)
plt.tight_layout()
plt.show()

print(f'{'Sample Size':>12} | {'Est. mu':>10} | {'Est. sigma':>12} | {'mu Error%':>10} | {'sigma Error%':>12}')
print('-' * 65)
for n, m, s in zip(sample_sizes, mle_mus, mle_sigmas):
    print(f'{n:>12} | {m:>10.4f} | {s:>12.4f} | {abs(m-true_mu)/true_mu*100:>9.4f}% | {abs(s-true_sigma)/true_sigma*100:>11.4f}%')

---
## Summary

### What We Did
1. Generated 1000 synthetic samples from N(50, 10)
2. Defined the Negative Log-Likelihood (NLL) function
3. Visualised the NLL landscape to understand what MLE is optimising
4. Used L-BFGS-B numerical optimisation to find MLE parameters
5. Verified numerical MLE = analytical MLE (sample mean & std)
6. Showed MLE consistency: estimation improves with sample size

### MLE in Generative AI — The Big Picture

| Concept | Classical Stats | Generative AI |
|---|---|---|
| **Parameters** | mu, sigma of a Gaussian | Billions of neural network weights |
| **Likelihood** | Normal PDF | Softmax probability of next token |
| **Objective** | Maximise log-likelihood | Minimise cross-entropy loss |
| **Optimiser** | L-BFGS-B (second-order) | Adam, AdamW (first-order, scalable) |
| **Data** | 1000 synthetic samples | Trillions of text/image tokens |
| **Consistency** | More data → better mu, sigma | More data → better generation quality |

> **Bottom Line:** MLE is the mathematical heart of training every modern Generative AI model. Understanding it at this level gives you the foundation to understand why cross-entropy loss is used, why more data helps, and why overfitting occurs when the model memorises rather than generalises.